In [ ]:
from scipy.signal import *# required for waveform analysis
import matplotlib.pyplot as plt # required to plot results
import numpy as np # required for calculation
import csv # required to handle csv files
import pandas as pd # required to handle dataframe for input/output data
#import tkinter as tk # required for dialog to select input file
#from tkinter import filedialog # required for dialog to select input file
#import scipy
#print(scipy.__version__)

dir = 'xxx' # define initial directory for the dialog
datalist=pd.read_csv('xxxx.csv') # read csv data
pixel_length = 6.8# length for 1 pixel
for j in range(len(datalist)):#iterative waveform analysis for all data listed in the csv file
    #datafile = dir + datalist['name'][j] + '_Results.csv'#open data file
    datafile = dir + datalist['name'][j] + '.csv'#open data file
    frame_rate = datalist['rate'][j]#read frame_rate
    data=pd.read_csv(datafile) # read csv output data
        
    datalength=data['Slice'].max()#define length of the time course as number of rows
    profile=np.zeros(datalength) # allocate vector for wave profile
    for i in range(datalength-1):# read oscillation profile
        profile[i+1]=data['dX'][i]#np.sign(data['dX'][i])*np.sqrt(data['dX'][i]**2+data['dY'][i]**2)#calculate Eucridian distance
    profile=(profile - profile.min()) * pixel_length# set minimum as zero and scale as physical length
    
    original = profile.copy() # store original profile
    time = np.array(range(0,datalength))#convert time index to numpy array
    time = time / frame_rate#scale to real time
    
    peaks, _ = find_peaks(profile, height=profile.mean()) # detect peak position
    peak_maxima = np.zeros(len(peaks)) # allocate vector for peak position
    peak_maxima = peaks  # obtain peak positions (frame)
    if len(peaks) - 1 > 0 :
        peak_interval = np.zeros(len(peaks)-1)
    else:
        peak_interval = np.nan
    #results_full = peak_widths(profile, peaks, rel_height=1, prominence_data=None, wlen= 25) # obtain width, start position, and finish position of peaks
    results_full = peak_widths(profile, peaks, rel_height=1, prominence_data=None, wlen= 150) # wlen was changed from 25 to 150 on Nov 1, 2023 by Akatsuki # obtain width, start position, and finish position of peaks
    results_full = np.array(results_full) / frame_rate#scale frame to real time
    results_full[1] = results_full[1] * frame_rate#recover to number of frames to draw lines in the plot
    results_full = list(results_full) # convert tuple to list
    
    plt.plot(time, profile) # plot profile
    plt.plot(peak_maxima / frame_rate, profile[peak_maxima], "x") # plot peak position with marker, x
    plt.hlines(*results_full[1:], color="C3") # show peak widths at basal level with horizontal lines
    
    dim = np.shape(results_full) # obtain dimension of peak widths data
    peak_width = np.zeros(dim[1]) # allocate vector to store peak widths
    peak_basal = np.zeros(dim[1]) # allocate vector to store basal of peaks
    peak_ini = np.zeros(dim[1]) # allocate vector for positions where contraction initiated
    peak_end = np.zeros(dim[1]) # allocate vector for positions where relaxation finished
    
    peak_width = results_full[0] # obtain peak width information
    peak_basal = results_full[1] # obtain basal of peaks
    peak_ini = results_full[2] # obtain contraction initiation positions (seconds)
    peak_end = results_full[3] # obtain relaxation finished positions (seconds)
    
    datafile=datafile.replace(".csv","")#remove file extension '.csv' from input data file name for prefix of output file names
    plt.savefig(datafile + '.png') # save plot as png file
    plt.cla()# clear plot axis
    
    cont = peak_maxima / frame_rate - peak_ini # calculate contraction periods (seconds)
    relax = peak_end - peak_maxima / frame_rate # calculate relaxation periods (seconds)
    peak_ini2 = peak_ini * frame_rate# replicate peak_ini converting real time to frames (frames) for drawing
    amp = original[peak_maxima.astype(int)] - original[peak_ini2.astype(int)]# calculate maximum amplitudes (micrometer)
    Results=pd.DataFrame([amp, cont, relax], ['amplitude', 'contract', 'relax']) # convert results for dataframe
    Results.T.to_csv(datafile + '_wave_form.csv') # write result to csv file
    if ~np.isnan(np.any(peak_interval)):#perform calculation if number of peaks is larger than 1
        for i in range(len(peaks)-1):
            peak_interval[i] = (peak_maxima[i+1] - peak_maxima[i]) / frame_rate #convert peak interval (seconds)
        Results=pd.DataFrame([peak_interval], ['peak interval']) # convert results for dataframe
        Results.T.to_csv(datafile + '_peak_interval.csv') # write result to csv file
    
#root.destroy() # close blank window